<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Table of Contents:

- [Imports](#imports)
- [Machine Learning Model Selection and Definition](#machine-learning-model-selection-and-definition)
- [Neural Network Model Selection and Definition](#neural-network-model-selection-and-definition)
- [Normal Data Modeling](#normal-data-modeling)
- [Complex Data Modeling](#complex-data-modeling)
- [Normal vs. Complex Data](#normal-vs.-complex-data)

</div>

### ORIENTATION:

This file is intended to select and define-as closely as possible-a Machine Learning (ML) and Neural Network (NN) model in order to train and evaluate the performace with the normal tabular data vs. the complex network feature engineered data.

---

### IMPORTANT NOTE:

yay

---

### SUMMARY:
- ML Selection
    - We will use *Random Forest Classifier* to maintain as much explainability and generally solve the data imbalance in the complex data
- NN Selection
    - We will create a simple *forward-pass* NN model with tensorflow and keras
- Normal Data Evaluation
    - yay
- Complex Data Evaluation
    - yay
- Normal vs. Complex Evaluation
    - yay

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Imports

- [Back to Table of Contents](#table-of-contents)

</div>

In [11]:
# File Detection
import os

# Databasing
import numpy as np
import pandas as pd

# Networking
import networkx as nx
import igraph as ig

# Database splitting, encoding, scaling
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, models


# Visualizations
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

# Timing
from time import time

# Matrix Manipulation
from scipy.sparse.linalg import eigsh

2026-03-09 09:55:36.116734: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Machine Learning Model Selection and Definition

- [Back to Table of Contents](#table-of-contents)

</div>

### SUMMARY:
- We will use *Random Forest Classifier* to maintain as much explainability and generally solve the data imbalance in the complex data

---

### DISCUSSION:

Generally in the realm of Artificial Intelligence, Machine Learning (ML) models when compared with Neural Network (NN) models are primarily superior due to their explainability capabilities which is vital for deeper understanding of data and ultimately better decisionmaking.  Unfortunately, to keep things understandable for people, there's a lot of higher-level math and repetitive application of statistics observed in NNs - which is why they generally perform significantly better - that does not exist in ML models.  This is to say that the purpose of defining the best ML model for **BOTH** normal data and complex data training is primarily to observe if there's better performance with complex data training which in turn allows for more explainability of the dataset.

That being said, this dataset is a classification problem so classification models make the obvious sense to implement.  The initial assumption is that the *Decision Tree Classifier* (DTC) would be the ideal ML for comparison;  However, in `explore_complex_networks.ipynb`, we observed that the distribution of `benign` attacks became well over 99% of the data.  This data imbalance is something that DTCs fail at preventing bias and for this reason, a sort of randomness in which features to explore and how would effectively make the model blind to this data imbalance.  Thankfully, the *Random Forest Classifer* (RFC) is does exactly this.

Thus, for ML comparison of normal data and complex data, we will implement the RFC.



In [12]:
def get_ml_model(trees:int=100) -> RandomForestClassifier:
    '''
    About
    -----
    - Creates and returns a basic sklearn RandomForestClassifier for model training

    Parameters
    ----------
    - trees (int) :
        - Default: 100
        - The number of decision trees to be created where each tree is randomly trained on a subset of the data.
          Essentially, this is like creating a voting block of whether or not something is significant during decisions

    Returns
    -------
    - RandomForestClassifier
    '''
    rfc_model = RandomForestClassifier(
        n_estimators=trees,      # This is just the number of "trees" we are creating
        class_weight='balanced', # This generally solves the imbalance issue
        max_features='sqrt',     # This generally solves the bias issue
        random_state=3703        # This is to ensure reproducibility
    )
    return rfc_model

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Neural Network Model Selection and Definition

- [Back to Table of Contents](#table-of-contents)

</div>

### SUMMARY:

- We will create a simple *forward-pass* NN model with tensorflow and keras

---

### DISCUSSION:

Inversely from the `Machine Learning Model Selection and Definition` discussion, NNs when compared with ML models are primarily superior in performance due to heavy reliance of higher-level math and repetitive statistics.  The biggest emphasis is the notion of *backpropagation* where it is best to visualize as a decision tree;  From some initial input, we incrementally make decisions until we arrive at the destination and in this case, return a prediction of what the answer is.  However, unlike decisions tree where this is the end of the process, *backpropagation* will take a wrong answer as a sign to walk backwards in all their decisions and adjust *weights and biases* for every decision until the correct answer is arrived at.  This is conducted as many times as is defined in the NN model structure with other parameters which is generally why NNs are notorious for being a *black box* in the sense of being impossible to explain their decisions and interpret them as key points of the dataset.  To emphasize, because so much higher-level math and repetitive statistical weighting and biasing, NNs lose much of their explainability, but generally performs significantly better than ML models.  Basically, it brute forces a pattern in the dataset regardless if that pattern actually means something in order to get the most correct answers as possible.

With this said, this section is to delinate NNs "brute forcing" patterns from the normal dataset and NNs "brute forcing" patterns that are mathematically created with complex network ideology from the complex dataset.  In other words, will the NN perform better without being told what underlying pattern(s) may or may not exist via the normal dataset or will the NN perform better by being told what underlying pattern(s) may or may not exist via the complex dataset.

Thus, for NN comparison, we will utilize tensorflow and keras libraries to define an **as basic of a model as possible** to prevent extreme overfitting on whatever patterns the NN thinks it sees.  Additionally, to maintain as close of comparison to the ML model selection as a *Random Forest Classifier*, we will define layers to implement a **forward-pass** concept.

In [ ]:
def get_nn_model(num_features_to_train:int) -> models.Sequential:
    '''
    About
    -----
    - Creates and returns a simple forward-pass Neural Network for model training
    - This definition is to closely resemble the RandomForestClassifer as much as possible via the forward-pass

    Parameters
    ----------
    - num_features_to_train (int) :
        - The number of features being used from the dataset to train the NN on

    Returns
    -------
    - models.Sequential
        - A tensorflow.keras NN model
    '''
    # ----- Define NN Structure -------------------------------------------------------------------
    nn_model = models.Sequential([

        # Input layer (Starting point of decision making)
        layers.Input(shape=(num_features_to_train,)),

        # Small hidden layers to prevent "brute force" memorization
        layers.Dense(32, activation='relu'),
        layers.Dense(16, activation='relu'),

        # Output layer (using Sigmoid for binary or Softmax for multiclass)
        layers.Dense(1, activation='sigmoid') 
    ])
    
    # ----- Define Backpropagation Methodology ----------------------------------------------------
    nn_model.compile(
        optimizer='adam',                       # How weights/biases work
        loss='binary_crossentropy',             # How significant was the incorrectness
        metrics=['accuracy',                    # The metrics to optimize
                 tf.keras.metrics.Precision(),
                 tf.keras.metrics.Recall()]
    )

    return nn_model

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Normal Data Modeling

- [Back to Table of Contents](#table-of-contents)

</div>

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Complex Data Modeling

- [Back to Table of Contents](#table-of-contents)

</div>

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Normal vs. Complex Data

- [Back to Table of Contents](#table-of-contents)

</div>